# Travel Agent Notebook

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Models

In [ ]:
from pydantic import BaseModel, Field

class BestOffer(BaseModel):
    """The subagent's answer: which offer it chose."""
    offer_id: str = Field(description="The exact offer_id string copied from a search-flights result. Never invent or reformat one.")
    reasoning: str = Field(description="One sentence on why this offer beats the alternatives.")

## State

In [3]:
from langchain.agents import AgentState

class TripState(AgentState):
    origin: str
    destination: str
    season: str
    departure_date: str
    return_date: str
    trip_length: str
    budget: str
    flights_offer: BestOffer


## Flights Agent

### Kiwi MCP Connection (search-flight)

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import shutil
import asyncio

is_live = os.environ.get("LIVE_MODE", "false").strip().lower() == "true"
if is_live:
    duffel_key = os.environ["DUFFEL_LIVE_API_KEY"]
else:
    duffel_key = os.environ["DUFFEL_TEST_API_KEY"]

client = MultiServerMCPClient(
    {
        "flights": {
            "transport": "stdio",
            "command": shutil.which("flights-mcp") or "flights-mcp",
            "args": [],
            "env": {"DUFFEL_API_KEY_LIVE": duffel_key},
        }
    }
)

flights_tools = await client.get_tools()
flights_tools_by_name = {t.name: t for t in flights_tools}


### Subagents

In [5]:
flights_agent_prompt = """
You are a flight-search specialist. Your goal is to find the cheapest flight options
for the given origin, destination and dates using the search-flights tool.

You can call get_offer_details to see more detailed information about flights.

You do not plan itineraries or book anything. You only look for flights.

When you have found the best offer, return the exact offer_id from the search 
results along with one sentence on why it beats the alternatives. Never invent an 
offer_id.
"""

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

flights_agent = create_agent(
    model="gpt-5-nano",
    tools=flights_tools,
    system_prompt=flights_agent_prompt,
    response_format=ToolStrategy(
        BestOffer,
        handle_errors="That didn't match the required format. Return the exact offer_id string from one of your search results."
        ),

)

## Orchestrator Tools

In [7]:
from langchain.messages import HumanMessage, ToolMessage
from tavily import TavilyClient
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command



tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def update_trip_info(
    runtime: ToolRuntime,
    origin: str | None = None,
    destination: str | None = None,
    season: str | None = None,
    departure_date: str | None = None,
    return_date: str | None = None,
    trip_length: str | None = None,
    budget: str | None = None
    ) -> Command:
    """Save any trip details the user has provided. Pass only the fields
    the user mentioned this turn and leave the rest as None."""

    updates = {k: v for k, v in {
        "origin": origin,
        "destination": destination,
        "season": season,
        "departure_date": departure_date,
        "return_date": return_date,
        "trip_length": trip_length,
        "budget": budget,
        "messages": [ToolMessage("Success", tool_call_id=runtime.tool_call_id)]
    }.items() if v is not None}
    
    return Command(update=updates)

@tool
async def call_flights_agent(runtime: ToolRuntime) -> str:
    """Call the flights subagent to find a flight offer."""

    missing = [k for k in ("origin", "destination", "departure_date", "return_date")
        if not runtime.state.get(k)]
    if missing:
        return f"Cannot search flights; missing: {', '.join(missing)}"
    
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    departure_date = runtime.state["departure_date"]
    return_date = runtime.state["return_date"]

    message = f"""
    Find round-trip flights from {origin} to {destination} departing
    on {departure_date} and returning on {return_date}
    """

    result = await flights_agent.ainvoke({"messages": [HumanMessage(content=message)]})

    ## Process the json response from the subagent and update the TripState to contain the Offer.
    pick = result.get("structured_response")

    if pick is None:
        return "The flights subagent finished without selecting an offer. Try the search again."

    return Command(update={"flights_offer": pick, "messages": [ToolMessage("Success", tool_call_id=runtime.tool_call_id)]})
    

@tool
async def get_saved_flights_info(runtime: ToolRuntime) -> str:
    """Gets detailed information about the flight offer currently saved in state."""
    
    offer_id = runtime.state.get("flights_offer").offer_id

    if offer_id is None:
        return """
        Can not get flight offer details; There is no flight offer saved in state.
        If you want to get a flight offer call the call_flights_agent tool first.
        """
    
    try:
        return await flights_tools_by_name["get_offer_details"].ainvoke({"params": {"offer_id": offer_id}})
    except Exception as e:
        return (
            f"Couldn't fetch details for offer {offer_id}: {e}."
            "The offer may have expired — consider re-running the flights search."
            )

## Orchestrator Prompt

In [ ]:
system_prompt = """
You are an expert travel agent. Help the user plan an amazing trip.
Always be friendly and keep responses concise and conversational.
Ask questions and find out more about the user's preferences and practical restraints
before proposing plans. However, do not bombard the user with multiple questions at once.

Use the web_search tool when needed to find up to date information on potential destinations. 

## Updating State
- If a user gives a trip detail (destination, season, dates, budget), call update_trip_info
to save it before replying.

## Flights
- If a user has decided on a destination and wants to find flights, use the call_flights_agent
tool to find a round-trip flight offer. This tool will search for flights and save the best offer
 in state. You can access full details of this offer by calling get_saved_flights_info.
- If the user has questions about the flight offer, call get_saved_flights_info to get detailed 
flight offer information.
"""

## Orchestrator Agent

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, update_trip_info, call_flights_agent, get_saved_flights_info],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
    state_schema=TripState
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "7"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="I want to visit Lisbon from October 10 to October 19. How much are flights from San Francisco?")]},
    config
    )

In [11]:
print(response["messages"][-1].content)

Here’s a current option from San Francisco (SFO) to Lisbon (LIS) for Oct 10–19, 2026:

- Price: $639.93 total per adult (Basic Economy)
  - Base fare: $542.31
  - Taxes/fees: $97.62
- Itinerary: 
  - Outbound: Oct 10, SFO 06:58 → LIS Oct 11 03:42 (nonstop, Iberia IB 3177), 12h44m
  - Return: Oct 19, LIS 13:56 → SFO 18:40 (nonstop, Iberia IB 3177), 12h44m
- Baggage: 1 checked bag + 1 carry-on included
- Fare rules: Change before departure with a $110 penalty; refund before departure has a $110 penalty
- Payment/hold: This saved offer shows a price/option window and a payment deadline; I can refresh to see current live options or try to hold this price if you want

Would you like me to lock in this fare for 1 adult in Economy, or should I search for more options (e.g., different times, nonstop only, or another carrier)?
